# Conformal Factuality

In [1]:
# %load_ext autoreload
# %autoreload 2
# %aimport -uqlm.scorers.longform.conformal

In [2]:
import numpy as np

from uqlm import LongTextCF
from uqlm.utils import (
    load_example_dataset,
    display_response_refinement,
    claims_dicts_to_lists,
    plot_model_accuracies,
)
from uqlm.longform import FactScoreGrader

<a id='section1'></a>
## 1. Set up LLM and Prompts

In this demo, we will illustrate this approach using the [FactScore](https://github.com/shmsw25/FActScore/tree/main/factscore) longform QA dataset. Alternatively, specify `factscore-stem-geo` for use of FactScore-STEM-Geo from [Bouchard et al., 2026](https://arxiv.org/abs/2602.17431). To implement with your use case, simply **replace the example prompts with your data**.  

In [3]:
# Load example dataset (FactScore)
factscore = load_example_dataset("factscore", n=5)[
    ["hundredw_prompt", "wikipedia_text"]
].rename(columns={"hundredw_prompt": "prompt"})
factscore.head()

# # Alternative dataset (FactScore-STEM-Geo)
# factscore_stem_geo = load_example_dataset("factscore-stem-geo", n=5)
# factscore_stem_geo.head()

Loading dataset - factscore...
Processing dataset...
Dataset ready!


,prompt,wikipedia_text
0,Tell me a bio of Suthida within 100 words.\n,Suthida Bajrasudhabimalalakshana (Thai: สมเด็จ...
1,Tell me a bio of Miguel Ángel Félix Gallardo w...,"Miguel Ángel Félix Gallardo (born January 8, 1..."
2,Tell me a bio of Iggy Azalea within 100 words.\n,"Amethyst Amelia Kelly (born 7 June 1990), know..."
3,Tell me a bio of Fernando da Costa Novaes with...,"Fernando da Costa Novaes (April 6, 1927 – Marc..."
4,Tell me a bio of Jan Zamoyski within 100 words.\n,Jan Sariusz Zamoyski (Latin: Ioannes Zamoyski ...


In this example, we use `ChatVertexAI` to instantiate our LLM, but any [LangChain Chat Model](https://js.langchain.com/docs/integrations/chat/) may be used. Be sure to **replace with your LLM of choice.**

In [4]:
# from langchain_google_vertexai import ChatVertexAI

# gemini_flash = ChatVertexAI(model="gemini-2.5-flash")
# gemini_flash_lite = ChatVertexAI(model="gemini-2.5-flash-lite")

# from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_ollama import ChatOllama

gemini_flash = ChatOllama(model="qwen2.5:7b") # "llama3.2")

<a id='section2'></a>
## 2. Generate LLM Responses and Claim/Sentence-Level Confidence Scores

In [5]:
claimcf = LongTextCF(
    llm=gemini_flash,
    claim_decomposition_llm=gemini_flash,
    frequency_scorer_llm=gemini_flash,
    scorers=["noncontradiction", "cosine_sim"],
    n_frequency_responses=2
    # max_calls_per_min=1000,
)

claim_filtering_scorer is not specified for response_refinement. Defaulting to noncontradiction.


In [6]:
factscore.prompt.to_list()

['Tell me a bio of Suthida within 100 words.\n',
 'Tell me a bio of Miguel Ángel Félix Gallardo within 100 words.\n',
 'Tell me a bio of Iggy Azalea within 100 words.\n',
 'Tell me a bio of Fernando da Costa Novaes within 100 words.\n',
 'Tell me a bio of Jan Zamoyski within 100 words.\n']

In [7]:
results = await claimcf.generate_and_score(
    prompts=factscore.prompt.to_list()[:2], response_refinement_threshold=0.85,
    show_progress_bars=True
)

🤖 Generation                                                                                                  
    - Generating responses...                                ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 0:00:09
                                                                                                                 
  ✂️ Decomposition                                                                                               
    - Decomposing responses into claims...                   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 0:00:16
   - Reconstructing responses with high-confidence claims... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 0:00:09

In [8]:
result_df = results.to_df()
result_df.head(5)

,prompt,response,claims_data,frequency_responses_list,refined_response,noncontradiction,cosine_sim
0,Tell me a bio of Suthida within 100 words.\n,"Suthida, born in Bangkok, Thailand, is an acco...","[{'claim': 'Suthida was born in Bangkok, Thail...","[Suthida, born in Bangkok, Thailand, is a prom...","Suthida was born in Bangkok, Thailand. This fa...",0.998387,0.870042
1,Tell me a bio of Miguel Ángel Félix Gallardo w...,"Miguel Ángel Félix Gallardo, born in 1945, was...",[{'claim': 'Miguel Ángel Félix Gallardo was bo...,"[Miguel Ángel Félix Gallardo, born in 1942, is...",Miguel Ángel Félix Gallardo was a prominent Me...,0.998026,0.967218


In [9]:
e

NameError: name 'e' is not defined

#### Response refinement

To illustrate how the response refinement operates, let's view an example. We first view the fine-grained claim-level data, including the claims in the original response, the claim-level confidence scores, and whether each claim was removed during the response refinement process. 

In [10]:
# View fine-grained claim data for a response
result_df.claims_data[0]

[{'claim': 'Suthida was born in Bangkok, Thailand.',
  'removed': False,
  'aggregated_score': 1.0,
  'frequency_score': 1.0},
 {'claim': 'Suthida is an accomplished artist.',
  'removed': True,
  'aggregated_score': 0.0,
  'frequency_score': 0.0},
 {'claim': "Suthida's paintings are vibrant and colorful.",
  'removed': True,
  'aggregated_score': 0.0,
  'frequency_score': 0.0},
 {'claim': "Suthida's paintings are abstract.",
  'removed': True,
  'aggregated_score': 0.0,
  'frequency_score': 0.0},
 {'claim': 'Suthida has a background in art education.',
  'removed': True,
  'aggregated_score': 0.0,
  'frequency_score': 0.0},
 {'claim': 'Suthida exhibits her works internationally.',
  'removed': True,
  'aggregated_score': 0.0,
  'frequency_score': 0.0},
 {'claim': 'Suthida gains recognition for her unique style.',
  'removed': True,
  'aggregated_score': 0.0,
  'frequency_score': 0.0},
 {'claim': "Suthida's style blends traditional Thai aesthetics with contemporary expressionism.",
  '

We can examine a particular claim in the response that was removed because its confidence score was too low. Let's see how this is reflected in the original vs. the refined response. 

In [11]:
display_response_refinement(
    original_text=result_df.response[0],
    claims_data=result_df.claims_data[0],
    refined_text=result_df.refined_response[0],
)

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Response Refinement Example

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── Original Response ───────────────────────────────────────────────╮
│ Suthida, born in Bangkok, Thailand, is an accomplished artist known for her vibrant and colorful abstract       │
│ paintings. With a background in art education, she has exhibited her works internationally, gaining recognition │
│ for her unique style that blends traditional Thai aesthetics with contemporary expressionism. Suthida's passion │
│ lies in exploring the interplay of colors and shapes to evoke emotions and inspire reflection among viewers.    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────── Low-Confidence Claims to be Removed ──────────────────────────────────────╮
│ • Suthida is an accomplished artist.                                                                            │
│ • Suthida's paintings are vibrant and colorful.                                                                 │
│ • Suthida's paintings are abstract.                                                                             │
│ • Suthida has a background in art education.                                                                    │
│ • Suthida exhibits her works internationally.                                                                   │
│ • Suthida gains recognition for her unique style.                                                               │
│ • Suthida's style blends traditional Thai aesthetics with contemporary expressionism.                           │
│ • Suthida explores the interplay of colors and shapes.                                                          │
│ • Suthida evokes emotions through her paintings.                                                                │
│ • Suthida inspires reflection among viewers.                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Refined Response ────────────────────────────────────────────────╮
│ Suthida was born in Bangkok, Thailand. This fact establishes her place of birth in the bustling capital city of │
│ Thailand, setting the stage for any further details that might be provided about her life and background.       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<a id='section3'></a>
## 3. Evaluate Hallucination Detection Performance

To evaluate hallucination detection performance, we 'grade' the atomic claims in the responses against an answer key. Here, we use UQLM's out-of-the-box `FactScoreGrader`, which can be used with [LangChain Chat Model](https://js.langchain.com/docs/integrations/chat/). **If you are using your own prompts/questions, be sure to update the grading method accordingly**.

In [12]:
# set up the LLM grader
grader = FactScoreGrader(llm=gemini_flash)

Before grading, we need to have claims formatted in list of lists where each interior list corresponds to a generated response. 

In [13]:
# Convert claims to list of lists
claims_data_lists = claims_dicts_to_lists(result_df.claims_data.tolist())

In [14]:
# grade original responses against the answer key using the grader
result_df["claim_grades"] = await grader.grade_claims(
    claim_sets=claims_data_lists["claim"], answers=factscore["wikipedia_text"].to_list()
)
result_df["answer"] = factscore["wikipedia_text"]
result_df.head(5)

,prompt,response,claims_data,frequency_responses_list,refined_response,noncontradiction,cosine_sim,claim_grades,answer
0,Tell me a bio of Suthida within 100 words.\n,"Suthida, born in Bangkok, Thailand, is an acco...","[{'claim': 'Suthida was born in Bangkok, Thail...","[Suthida, born in Bangkok, Thailand, is a prom...","Suthida was born in Bangkok, Thailand. This fa...",0.998387,0.870042,"[False, False, False, False, False, False, Fal...",Suthida Bajrasudhabimalalakshana (Thai: สมเด็จ...
1,Tell me a bio of Miguel Ángel Félix Gallardo w...,"Miguel Ángel Félix Gallardo, born in 1945, was...",[{'claim': 'Miguel Ángel Félix Gallardo was bo...,"[Miguel Ángel Félix Gallardo, born in 1942, is...",Miguel Ángel Félix Gallardo was a prominent Me...,0.998026,0.967218,"[False, True, True, True, True, False, False, ...","Miguel Ángel Félix Gallardo (born January 8, 1..."


In [15]:
all_claim_scores, all_claim_grades = [], []
for i in range(len(result_df)):
    all_claim_scores.extend(claims_data_lists["noncontradiction"][i])
    all_claim_grades.extend(result_df["claim_grades"][i])

print(f"""Baseline LLM accuracy: {np.mean(all_claim_grades)}""")

KeyError: 'noncontradiction'

#### 3.1 Claim-Level Hallucination Detection AUROC

To evaluate fine-grained hallucination detection performance, we compute AUROC of claim-level hallucination detection. Below, we plot the ROC curve and report these results.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_true=all_claim_grades, y_score=all_claim_scores)
roc_auc = roc_auc_score(y_true=all_claim_grades, y_score=all_claim_scores)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.show()

#### 3.2 Gains from Uncertainty-Aware Decoding

Lastly, we evaluate the gains from uncertainty-aware decoding (UAD) by measuring the factual precision over claims at various filtering thresholds. 

In [ ]:
plot_model_accuracies(
    scores=all_claim_scores,
    correct_indicators=all_claim_grades,
    title="LLM Accuracy by Claim Confidence Threshold",
    display_percentage=True,
)

Since, we have selected a threshold of 0.85, we can measure LLM accuracy with and without UAD.

In [ ]:
thresh = 0.85
filtered_grades, filtered_scores = [], []
for grade, score in zip(all_claim_grades, all_claim_scores):
    if score > thresh:
        filtered_grades.append(grade)
        filtered_scores.append(score)

print(f"Baseline LLM factual precision: {np.mean(all_claim_grades)}")
print(f"UAD-Improved LLM factual precision: {np.mean(filtered_grades)}")

<a id='section4'></a>
## 4. Scorer Definitions

© 2025 CVS Health and/or one of its affiliates. All rights reserved.